# Solutions · 02 · Conversion and reactor sizing: Levenspiel plots

Worked solutions to the exercises of [notebook 02](../notebooks/02_conversion_and_reactor_sizing.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, integrate
from engreact import ideal, rate_data
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
X = np.array([0, 0.1, 0.2, 0.4, 0.6, 0.7, 0.8])
rate = np.array([0.45, 0.37, 0.30, 0.195, 0.113, 0.079, 0.05])
F_A0 = 0.4
rate_fn = lambda x: np.interp(x, X, rate)

## Exercise 1

Using the first rate table, find the minimum total volume of two CSTRs in series reaching X = 0.8 by scanning the intermediate conversion, and compare with a single CSTR and with a PFR.

In [3]:
best = None
for X1 in np.linspace(0.05, 0.75, 141):
    V1 = F_A0 * X1 / rate_fn(X1)
    V2 = F_A0 * (0.8 - X1) / rate_fn(0.8)
    if best is None or V1 + V2 < best[0]:
        best = (V1 + V2, X1, V1, V2)
print(f"two CSTRs: minimum total {best[0]:.2f} m3 with the first to X = {best[1]:.2f} ({best[2]:.2f} + {best[3]:.2f} m3)")
print(f"single CSTR: {F_A0 * 0.8 / rate_fn(0.8):.2f} m3; PFR: {ideal.levenspiel(X, rate, F_A0)['V_pfr'][-1]:.2f} m3")

two CSTRs: minimum total 3.65 m3 with the first to X = 0.55 (1.65 + 2.00 m3)
single CSTR: 6.40 m3; PFR: 2.20 m3


Splitting the CSTR duty into two stages nearly halves the total volume; the optimum intermediate
conversion is where the two rectangles have equal "height gain" - Levenspiel's maximisation of the
rectangle area. Two stages capture most of the benefit; the PFR is the limit of infinitely many.

## Exercise 2

Fit the first rate table to -r_A = k C_A (C_A = C_A0 (1 - X)) and recompute the PFR and CSTR volumes from the rate law instead of the table. How much do they differ from the trapezoidal areas, and why?

In [4]:
C_A = 2500.0 * (1 - X)
fit = rate_data.initial_rates_method(C_A[1:], rate[1:])
n, k = fit["order"], fit["k"]
rate_law = lambda x: k * (2500.0 * (1 - x)) ** n
Vp, Vc = ideal.pfr_volume(F_A0, 0.8, rate_law), ideal.cstr_volume(F_A0, 0.8, rate_law)
lev = ideal.levenspiel(X, rate, F_A0)
print(f"power-law fit: order {n:.2f}; PFR {Vp:.2f} m3 (table: {lev['V_pfr'][-1]:.2f}), CSTR {Vc:.2f} m3 (table: {lev['V_cstr'][-1]:.2f})")

power-law fit: order 1.33; PFR 2.11 m3 (table: 2.20), CSTR 6.77 m3 (table: 6.40)


The rate law and the table agree within a few percent for the PFR (the fitted curve smooths the trapezoids)
and exactly-ish for the CSTR, which uses only the last point. The differences measure the interpolation
error of a 7-point table - and the fit lets one extrapolate beyond X = 0.8, which the table cannot.

## Exercise 3

**Implement it yourself:** write `cstr_series(F_A0, volumes, rate_fn)` that solves each stage's balance with brentq and reproduces `ideal.reactors_in_series` for two CSTRs.

In [5]:
def cstr_series(F_A0, volumes, rate_fn):
    X_in, out = 0.0, []
    for V in volumes:
        X_out = optimize.brentq(lambda x: F_A0 * (x - X_in) / rate_fn(x) - V, X_in + 1e-9, 0.999)
        out.append(X_out); X_in = X_out
    return out
print("by hand:", np.round(cstr_series(F_A0, [1.15, 1.15], rate_fn), 4), "; library:", [round(s["X_out"], 4) for s in ideal.reactors_in_series(F_A0, [("cstr", 1.15), ("cstr", 1.15)], rate_fn)])

by hand: [0.4737 0.7005] ; library: [0.4737, 0.7005]


Each stage's balance uses the conversion leaving the previous one as its inlet; a root find per stage is all
there is to a cascade.